# Overfitting and Regularisation

A neural network has many weights, often far more than there are training examples. That gives it the flexibility to learn complicated patterns, but also the capacity to memorise its training data, noise included. In this notebook you recognise overfitting in the learning curves, apply the three most common remedies in Keras (L2 regularisation, dropout and early stopping) and save the best model of a training run with a checkpoint.

## Learning Objectives

By the end of this notebook, you should be able to:

- Diagnose underfitting, overfitting and a good fit from training and validation learning curves.
- Compare a small and a large network on the same data by their validation RMSE.
- Add L2 regularisation, dropout and early stopping to a Keras model.
- Save the best epoch of a training run with a `ModelCheckpoint` callback and load it again.
- Select a model on the validation set and evaluate it once on the test set.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # fix the random seed so every run gives the same data

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## 1. Reading Learning Curves

A **learning curve** plots the loss (or another metric) after every epoch, once for the training set and once for the validation set. The shape of the two curves, and the distance between them, tells you what to change next.

Look at the six pairs of curves below before you read on. Which panel shows overfitting? In which one should the network train for longer? Which one suggests that the validation set is too small or too easy?

<img src="../assets/learning_curves_quiz.png" width="900">

<details>
<summary>Show the diagnosis</summary>

<img src="../assets/learning_curves.png" width="900">

- **A. Good fit**: both losses level off at a low value, close together. Keep the model.
- **B. Underfitting**: both losses level off at a high value. The model is too simple: add layers or neurons, or train longer.
- **C. Overfitting**: the training loss keeps falling while the validation loss reaches a minimum and rises again. The model is learning the noise of the training set. Reduce its capacity, get more data or regularise it (this notebook).
- **D. Still improving**: both losses are still falling when training stops. Train for more epochs.
- **E. Learning rate too high**: the losses jump from epoch to epoch instead of falling smoothly. Lower the learning rate.
- **F. Unrepresentative validation set**: the validation loss stays below the training loss. The validation set is probably too small or easier than the training set; check how it was split. Dropout (section 5) can cause the same pattern, because the training loss is computed with neurons switched off.

</details>

## 2. The Data

To see overfitting clearly, the data here is generated from a known curve, $f(x) = \sin(1.5x) + 0.3x$, with random noise of standard deviation 0.35 added to every target. The training set has only 40 examples, few enough for a large network to memorise. Because the curve is known, you can draw it next to every model's predictions.

The noise also sets a lower limit: even the true curve has an RMSE of about 0.35 on new data, because the noise cannot be predicted. A validation RMSE close to 0.35 is therefore close to the best any model can do.

In [ ]:
def true_curve(x):
    """The curve the data is generated from."""
    return np.sin(1.5 * x) + 0.3 * x


def make_data(n_examples):
    """Return n_examples points with x between -3 and 3 and a noisy target y."""
    x = rng.uniform(-3, 3, size=n_examples)
    return pd.DataFrame({"x": x, "y": true_curve(x) + rng.normal(0, 0.35, size=n_examples)})


train = make_data(40)
valid = make_data(200)
test = make_data(1000)

X_train, y_train = train[["x"]], train["y"]
X_valid, y_valid = valid[["x"]], valid["y"]
X_test, y_test = test[["x"]], test["y"]
print(f"Training: {len(train)}, validation: {len(valid)}, test: {len(test)} examples")

In [ ]:
x_grid = pd.DataFrame({"x": np.linspace(-3, 3, 400)})  # evenly spaced inputs for drawing curves
plt.scatter(train["x"], train["y"], s=18, label="training examples")
plt.plot(x_grid["x"], true_curve(x_grid["x"]), "k--", label="true curve")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

### Helper Functions

`build_model` creates a new regression network. Its arguments are the settings you change in this notebook: the hidden layers, the strength of L2 regularisation (section 4) and the dropout rate (section 5). `train_model` compiles and trains it with Adam. With only 40 training examples, `batch_size=40` makes every update use the whole training set: one update per epoch, like the batch gradient descent in the table of notebook 01.

`show_run` draws two plots for a trained model: its predictions next to the true curve, and its learning curves. It also stores three numbers in the `results` dictionary for the comparison in section 8: the lowest validation RMSE during training, the epoch where it was reached, and the validation RMSE of the model at the end of training.

In [ ]:
def build_model(units, l2_strength=0.0, dropout_rate=0.0):
    """Return a new regression network with ReLU hidden layers, optional L2 regularisation and dropout."""
    keras.utils.clear_session()  # start from a clean Keras state
    keras.utils.set_random_seed(42)  # the same starting weights for the same settings
    regularizer = keras.regularizers.L2(l2_strength) if l2_strength > 0 else None
    layers = [keras.Input(shape=(1,))]
    for n_neurons in units:
        layers.append(keras.layers.Dense(units=n_neurons, activation="relu", kernel_regularizer=regularizer))
        if dropout_rate > 0:
            layers.append(keras.layers.Dropout(dropout_rate))
    layers.append(keras.layers.Dense(units=1))  # regression output: one neuron, no activation
    return keras.Sequential(layers)


def train_model(model, epochs=500, callbacks=None):
    """Compile and train a model with Adam; return its learning curves as a DataFrame."""
    model.compile(
        loss="mse",
        optimizer=keras.optimizers.Adam(learning_rate=0.01),
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    history = model.fit(
        X_train, y_train, epochs=epochs, batch_size=40, validation_data=(X_valid, y_valid),
        callbacks=callbacks, verbose=0,
    )
    return pd.DataFrame(history.history)


results = {}


def show_run(name, model, logs):
    """Plot the predictions and the learning curves of a trained model and store its validation RMSEs."""
    fig, (ax_fit, ax_curves) = plt.subplots(1, 2, figsize=(12, 4))
    ax_fit.scatter(train["x"], train["y"], s=18, label="training examples")
    ax_fit.plot(x_grid["x"], true_curve(x_grid["x"]), "k--", label="true curve")
    ax_fit.plot(x_grid["x"], model.predict(x_grid, verbose=0), color="tab:red", linewidth=2, label="prediction")
    ax_fit.set(xlabel="x", ylabel="y", title=f"{name}: predictions")
    ax_fit.legend()
    ax_curves.plot(logs["rmse"], label="training RMSE")
    ax_curves.plot(logs["val_rmse"], label="validation RMSE")
    ax_curves.axhline(0.35, color="grey", linestyle=":", label="noise level (0.35)")
    ax_curves.set(xlabel="Epoch", ylabel="RMSE", ylim=(0, 1), title=f"{name}: learning curves")
    ax_curves.legend()
    plt.show()
    results[name] = {
        "lowest validation RMSE": logs["val_rmse"].min(),
        "epoch of the lowest": logs["val_rmse"].idxmin() + 1,  # epochs are counted from 1
        "validation RMSE at the end": model.evaluate(X_valid, y_valid, verbose=0)[1],
    }
    return pd.Series(results[name]).round(3)

## 3. Model Size

The simplest way to control overfitting is the size of the network. The cells below train a **small** network, one hidden layer of 8 neurons, and a **large** one, three hidden layers of 256 neurons, for 500 epochs each.

In [ ]:
small_model = build_model(units=(8,))
small_model.summary()
show_run("Small network", small_model, train_model(small_model))

In [ ]:
large_model = build_model(units=(256, 256, 256))
large_model.summary()
show_run("Large network", large_model, train_model(large_model))

The small network has 25 weights and biases, fewer than the 40 training examples. Its prediction is made of a few straight pieces that follow the true curve, and its validation RMSE falls slowly and steadily to 0.368, close to the noise level of 0.35. Its training RMSE ends at 0.24. Both curves are still falling slightly at epoch 500, like panel D.

The large network has 132,353 weights and biases, more than 3,000 for every training example. Its prediction follows individual training points, for example the peaks and dips around $x = 1.5$, which are noise rather than part of the true curve. The learning curves show the same thing as panel C: the training RMSE keeps falling to about 0.15, far below the noise level, while the validation RMSE reaches its lowest value, 0.367, at epoch 42 and then rises slowly to 0.407. The gap between training and validation RMSE, 0.15 against 0.41, is much wider than for the small network (0.24 against 0.37): the large network has learnt details of the training set that do not generalise to new data.

## 4. L2 Regularisation

You met L2 regularisation earlier in the course with Ridge regression. It works the same way in a network: a penalty proportional to the sum of the squared weights is added to the loss,

$$\text{loss} = \text{MSE} + \lambda \sum_i w_i^2,$$

so gradient descent keeps the weights small unless a large weight clearly lowers the MSE. Small weights give a smoother function, which cannot follow every noisy point. The strength $\lambda$ is a hyperparameter: too small and nothing changes, too large and the network underfits.

In Keras, L2 regularisation is set per layer with the `kernel_regularizer` argument and a [`keras.regularizers.L2`](https://keras.io/api/layers/regularizers/) object; `build_model` adds it to every hidden layer. Keras adds the penalty to the loss it reports, for the training and the validation set, so the loss of a regularised model is not comparable with the loss of an unregularised one. This notebook therefore compares models by their RMSE, which does not include the penalty. The cell below trains the large network again with $\lambda = 0.01$.

In [ ]:
l2_model = build_model(units=(256, 256, 256), l2_strength=0.01)
show_run("Large network with L2", l2_model, train_model(l2_model))

With L2 regularisation the prediction no longer follows single training points: like the small network's, it is made of a few straight pieces close to the true curve. The training RMSE stays at about 0.24 instead of falling to 0.15, and the validation RMSE ends at 0.369 instead of 0.407. The penalty has removed most of the overfitting while the network keeps all 132,353 weights.

### The Jumps in the Learning Curves

Both curves of the L2 run show a sudden increase, a jump, several times, for example around epochs 85, 315 and 370, where the validation RMSE rises within one or two epochs from about 0.37 to about 0.5 and then falls back. The large network without regularisation shows a smaller jump near epoch 250. These jumps come from the optimiser, not from the data or the regularisation:

- Every update here uses all 40 training examples, so once the loss has flattened out the gradients become very small.
- Adam divides each step by a running average of the recent gradient sizes (section 3 of notebook 01). When the gradients stay small for a while, this average shrinks too, so the steps stay large relative to the gradients.
- Occasionally one of these steps is too large and the loss increases sharply. The larger gradients that follow reduce the step size again, and the loss returns to its previous level.

What to do about jumps like these:

- **Judge the trend, not a single epoch.** A jump that recovers within a few epochs does not change which model is better; compare the level of the curves over many epochs.
- **Keep the best epoch** with early stopping (section 6) or a checkpoint (section 7), so a jump near the end of training does not decide which weights you keep.
- **Lower the learning rate** if the jumps are large or do not recover, for example `keras.optimizers.Adam(learning_rate=0.001)`. The steps become smaller, so the jumps shrink, but training needs more epochs. Here the learning rate stays at 0.01 so that every model trains in a few seconds.

For further reading: the [`ReduceLROnPlateau`](https://keras.io/api/callbacks/reduce_lr_on_plateau/) callback lowers the learning rate during training once the validation loss stops improving, which combines large steps at the start with small steps at the end.

## 5. Dropout

A [`Dropout`](https://keras.io/api/layers/regularization_layers/dropout/) layer switches off a random share of the outputs of the layer before it at every training step; the share is the **dropout rate**. Because any neuron can be switched off, no neuron can rely on a particular other one, and the network has to distribute what it learns across many neurons. This makes it harder to memorise single examples.

Dropout is only active during training. When the model predicts or is evaluated, all neurons are used. This is why the training RMSE with dropout is higher and noisier than without it. The cell below adds dropout with a rate of 0.4 after every hidden layer of the large network.

In [ ]:
dropout_model = build_model(units=(256, 256, 256), dropout_rate=0.4)
show_run("Large network with dropout", dropout_model, train_model(dropout_model))

With dropout the training RMSE is noisy, because every epoch switches off a different set of neurons, and it stays at about 0.3. The prediction no longer follows single training points; between $x = 0.5$ and $x = 1.7$ it is almost flat, slightly below the peak of the true curve. The validation RMSE ends at 0.370, again much closer to the noise level than the 0.407 of the large network without regularisation.

## 6. Early Stopping

The learning curves of the large network show that its validation RMSE was lowest early in training. **Early stopping** uses this directly: it stops training once the validation loss has not improved for a set number of epochs (the **patience**), so the run ends that many epochs after the best one. With `restore_best_weights=True` it then returns to the weights of the best epoch.

In Keras, early stopping is a [callback](https://keras.io/api/callbacks/), an object that `fit()` calls after every epoch. [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) takes the value to watch (`monitor`), how many epochs without improvement to accept (`patience`) and whether to restore the best weights at the end (`restore_best_weights`).

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=50, restore_best_weights=True)
early_model = build_model(units=(256, 256, 256))
early_logs = train_model(early_model, callbacks=[early_stopping])
print(f"Training stopped after {len(early_logs)} of 500 epochs")
show_run("Large network with early stopping", early_model, early_logs)

Training stopped after 92 epochs: the validation loss was lowest at epoch 42 and did not improve in the 50 epochs that followed. Because `restore_best_weights=True`, the model returned to the weights of epoch 42, and its validation RMSE is 0.367, the lowest value of the whole run. Early stopping also saves time: it trained for 92 epochs instead of 500.

## 7. Keep the Best Model with a Checkpoint

Early stopping ends training soon after the best epoch. Sometimes you want to let training run, for example to see whether the validation loss falls again later, and still keep the best model. A [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/) callback does this: it saves the model to a `.keras` file during training, and with `save_best_only=True` it overwrites the file only when the monitored value improves. After training, the file holds the model from the best epoch, which you load with [`keras.saving.load_model()`](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/#loadmodel-function), as you did in the Keras workflow notebook of the Artificial Neural Networks repository.

The cell below trains the large network without regularisation for all 500 epochs again, this time with a checkpoint.

In [ ]:
Path("models").mkdir(exist_ok=True)  # the checkpoint file is written to this folder
checkpoint = keras.callbacks.ModelCheckpoint(
    "models/best_large_network.keras", monitor="val_loss", save_best_only=True
)
last_model = build_model(units=(256, 256, 256))
checkpoint_logs = train_model(last_model, callbacks=[checkpoint])

checkpoint_model = keras.saving.load_model("models/best_large_network.keras")
print(f"Validation RMSE after epoch 500:     {last_model.evaluate(X_valid, y_valid, verbose=0)[1]:.3f}")
print(f"Validation RMSE of the checkpoint:   {checkpoint_model.evaluate(X_valid, y_valid, verbose=0)[1]:.3f}")

After 500 epochs the network has a validation RMSE of 0.407, as in section 3. The checkpoint was last overwritten at epoch 42, the epoch with the lowest validation loss, and the model loaded from it reaches 0.367, the same as the early-stopping model. The two approaches differ in one respect: early stopping saves training time, while a checkpoint lets training run to the end and still keeps the best model on disk. A checkpoint also protects a long training run, because the best model so far is saved even if training is interrupted.

## 8. Compare the Models on the Validation Set

All models were compared on the validation set, and early stopping and the checkpoint used it to pick their best epoch. The table collects the results.

In [ ]:
comparison = pd.DataFrame(results).T
comparison.round(3)

The lowest validation RMSE is almost the same for every model (0.364 to 0.368): at its best epoch, even the large network is a good model. The difference is in the last column. Without regularisation the large network ends at 0.407, because it keeps fitting noise after epoch 42. Each of the three techniques keeps the validation RMSE at the end between 0.367 and 0.370, close to the noise level of 0.35. On this data, the small network does equally well; on real problems with more complex patterns, a large regularised network usually does better than a small one.

## Try It: Combine the Techniques

The techniques can be combined. Build the large network with both L2 regularisation and dropout and train it with early stopping. Choose $\lambda$, the dropout rate and the patience yourself; the values used above are a good starting point. Because the loss of an L2 model includes the penalty, let early stopping monitor `"val_rmse"` instead of `"val_loss"`.

Aim for a validation RMSE at the end at or below the best value in the table, about 0.37. The test set is not used here: it is kept for section 9.

In [ ]:
my_l2_strength = 0.01
my_dropout_rate = 0.2
# the loss of an L2 model includes the penalty, so monitor the RMSE instead
my_early_stopping = keras.callbacks.EarlyStopping(monitor="val_rmse", patience=50, restore_best_weights=True)

combined_model = build_model(units=(256, 256, 256), l2_strength=my_l2_strength, dropout_rate=my_dropout_rate)
combined_logs = train_model(combined_model, callbacks=[my_early_stopping])
show_run("Combined", combined_model, combined_logs)
print(
    f"Validation RMSE at the end: {results['Combined']['validation RMSE at the end']:.3f} "
    f"(large network without regularisation: {results['Large network']['validation RMSE at the end']:.3f})"
)

**Result.** The solution combines L2 ($\lambda = 0.01$), dropout at a rate of 0.2 and early stopping on the validation RMSE with a patience of 50. Its validation RMSE at the end is 0.364, on the target and the lowest value in the comparison, against 0.407 for the large network without regularisation. Combining the techniques does not add up their effects here, because each one alone already brings the model close to the noise level of 0.35.

## 9. Final Check on the Test Set

Because the validation set was used to make these choices, its RMSE is slightly optimistic: the settings were chosen to perform well on it. The test set has not been used yet, so it gives an unbiased estimate for the chosen model. It is used once, here, for the chosen model, the large network with early stopping, which has the lowest validation RMSE at the end (the small network is only 0.001 behind). The large network without regularisation is evaluated as well, for reference.

In [ ]:
models = {
    "Small network": small_model,
    "Large network": large_model,
    "Large network with L2": l2_model,
    "Large network with dropout": dropout_model,
    "Large network with early stopping": early_model,
}
for name in ["Large network with early stopping", "Large network"]:
    test_rmse = models[name].evaluate(X_test, y_test, verbose=0)[1]
    print(f"{name}: test RMSE {test_rmse:.3f}")

On the test set, the model with early stopping reaches an RMSE of 0.373 and the large network without regularisation 0.411. The test RMSEs are close to the validation RMSEs (0.367 and 0.407), so the choice made on the validation set holds for new data. The 1,000 test examples were never used for a decision, so this comparison gives an unbiased estimate of how the models perform on new data.

## Summary

- Learning curves show the fit: both curves high means underfitting, a falling training curve with a rising validation curve means overfitting.
- A network with many more weights than training examples can memorise the noise in its training data. Making it smaller is the simplest remedy.
- L2 regularisation adds the squared weights to the loss and keeps the learnt function smooth. Dropout switches off random neurons during training, so no neuron can rely on another. Early stopping ends training at the epoch with the lowest validation loss.
- A `ModelCheckpoint` callback with `save_best_only=True` saves the model of the best epoch to a `.keras` file while training continues.
- All of these choices are made on the validation set, so the test set is used only once, for the final model.

Next, apply these techniques to real data in [03 - Exercise: Regularise the Forest Cover Classifier](03_exercise_forest_cover.ipynb).

## References

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd edition, chapter 11: Avoiding Overfitting Through Regularization.
- Chollet, F. and Watson, M. (2025). *Deep Learning with Python*, 3rd edition, chapter 5: Fundamentals of machine learning. [Free to read online](https://deeplearningwithpython.io/).
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. and Salakhutdinov, R. (2014). [Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://jmlr.org/papers/v15/srivastava14a.html). *Journal of Machine Learning Research*, 15, 1929-1958.
- Keras API: [layer weight regularisers](https://keras.io/api/layers/regularizers/), [`Dropout`](https://keras.io/api/layers/regularization_layers/dropout/), [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) and [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/).
- TensorFlow tutorial: [Overfit and underfit](https://www.tensorflow.org/tutorials/keras/overfit_and_underfit), which the earlier version of this notebook followed.